# OffScript — Phase 1: Initial Data Exploration

Pulling and exploring Statcast pitch-by-pitch data for a curated 
set of MLB pitchers across the 2023 and 2024 seasons.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pybaseball import statcast_pitcher, playerid_lookup
from pybaseball import cache

# Enable caching — prevents re-downloading data on every run
cache.enable()

print("Imports successful")

In [ ]:
# Look up Gerrit Cole's MLBAM ID
cole_lookup = playerid_lookup('Cole', 'Gerrit')
print(cole_lookup[['name_last', 'name_first', 'key_mlbam']])

In [ ]:
# Pull 2023 and 2024 Statcast data for Gerrit Cole
cole_id = 543037

print("Pulling Statcast data for Gerrit Cole...")
cole_data = statcast_pitcher('2023-03-30', '2024-11-01', cole_id)

print(f"Total pitches: {len(cole_data)}")
print(f"Seasons: {cole_data['game_date'].min()} to {cole_data['game_date'].max()}")
print(f"Columns available: {len(cole_data.columns)}")

In [ ]:
cols_of_interest = [
    'game_date',
    'pitcher',
    'player_name',
    'pitch_type',
    'pitch_name',
    'release_speed',
    'pfx_x', 'pfx_z',
    'plate_x', 'plate_z',
    'balls', 'strikes',
    'on_1b', 'on_2b', 'on_3b',
    'stand',
    'p_throws',
    'events',
    'description',
    'inning',
    'home_score',
    'away_score'
]

cole_trimmed = cole_data[cols_of_interest].copy()

# Create count state column
cole_trimmed['count'] = (cole_trimmed['balls'].astype(str) + '-' + 
                         cole_trimmed['strikes'].astype(str))

# Create score differential column
cole_trimmed['score_diff'] = cole_trimmed['home_score'] - cole_trimmed['away_score']

print(cole_trimmed.shape)
print(cole_trimmed.dtypes)

In [ ]:
# Pitch type distribution
print("=== Pitch Type Distribution ===")
print(cole_trimmed['pitch_type'].value_counts())

print("\n=== Count State Distribution ===")
print(cole_trimmed['count'].value_counts().sort_index())

print("\n=== Batter Handedness ===")
print(cole_trimmed['stand'].value_counts())

print("\n=== Missing Values ===")
print(cole_trimmed.isnull().sum()[cole_trimmed.isnull().sum() > 0])

In [ ]:
# Pitch selection by count state
count_pitch = (
    cole_trimmed.groupby(['count', 'pitch_type'])
    .size()
    .reset_index(name='n')
)
count_pitch['pct'] = (count_pitch.groupby('count')['n']
                      .transform(lambda x: x / x.sum() * 100))

fig, ax = plt.subplots(figsize=(14, 6))
pivot = (count_pitch.pivot(index='count', columns='pitch_type', values='pct')
         .fillna(0))
pivot.plot(kind='bar', stacked=True, ax=ax, colormap='tab10')

ax.set_title("Gerrit Cole — Pitch Selection by Count (2023-2024)", 
             fontsize=14, pad=15)
ax.set_ylabel("Usage %")
ax.set_xlabel("Count")
ax.legend(title="Pitch Type", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()

# Save to reports folder
plt.savefig('../reports/figures/cole_pitch_selection_by_count.png', 
            dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to reports/figures/")

In [ ]:
import matplotlib.patches as patches

def plot_pitch_locations(data, pitcher_name, pitch_types=None):
    if pitch_types:
        data = data[data['pitch_type'].isin(pitch_types)]
    
    pitches = data['pitch_type'].unique()
    n = len(pitches)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 6))
    
    if n == 1:
        axes = [axes]
    
    for ax, pitch in zip(axes, pitches):
        subset = data[data['pitch_type'] == pitch]
        
        # Strike zone box
        zone = patches.Rectangle(
            (-0.85, 1.5), 1.7, 2.0,
            linewidth=2, edgecolor='black', facecolor='none'
        )
        ax.add_patch(zone)
        
        # Plot pitch locations
        ax.scatter(subset['plate_x'], subset['plate_z'], 
                  alpha=0.15, s=10, color='steelblue')
        
        ax.set_xlim(-2.5, 2.5)
        ax.set_ylim(0, 5)
        ax.set_title(f"{pitch}\nn={len(subset)}")
        ax.set_xlabel("Horizontal Location")
        ax.set_ylabel("Vertical Location")
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
    
    fig.suptitle(f"{pitcher_name} — Pitch Locations by Type (2023-2024)", 
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(f'../reports/figures/{pitcher_name.lower().replace(" ", "_")}_locations.png',
                dpi=150, bbox_inches='tight')
    plt.show()

plot_pitch_locations(cole_trimmed, "Gerrit Cole")

In [ ]:
def plot_location_by_count(data, pitcher_name, counts=['0-0', '0-2', '3-0', '3-2']):
    fig, axes = plt.subplots(1, len(counts), figsize=(5 * len(counts), 6))
    
    for ax, count in zip(axes, counts):
        subset = data[data['count'] == count]
        
        zone = patches.Rectangle(
            (-0.85, 1.5), 1.7, 2.0,
            linewidth=2, edgecolor='black', facecolor='none'
        )
        ax.add_patch(zone)
        
        # Color by pitch type
        for pitch, color in zip(subset['pitch_type'].unique(), 
                                 plt.cm.tab10.colors):
            p = subset[subset['pitch_type'] == pitch]
            ax.scatter(p['plate_x'], p['plate_z'],
                      alpha=0.3, s=15, label=pitch, color=color)
        
        ax.set_xlim(-2.5, 2.5)
        ax.set_ylim(0, 5)
        ax.set_title(f"Count: {count}\nn={len(subset)}")
        ax.set_xlabel("Horizontal")
        ax.legend(fontsize=7)
    
    fig.suptitle(f"{pitcher_name} — Location by Count State", 
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(f'../reports/figures/{pitcher_name.lower().replace(" ", "_")}_location_by_count.png',
                dpi=150, bbox_inches='tight')
    plt.show()

plot_location_by_count(cole_trimmed, "Gerrit Cole")

In [ ]:
# Dylan Cease — extreme slider usage, good deviation candidate
cease_id = 656302

print("Pulling Statcast data for Dylan Cease...")
cease_data = statcast_pitcher('2023-03-30', '2024-11-01', cease_id)

cease_trimmed = cease_data[cols_of_interest].copy()
cease_trimmed['count'] = (cease_trimmed['balls'].astype(str) + '-' + 
                          cease_trimmed['strikes'].astype(str))
cease_trimmed['score_diff'] = (cease_trimmed['home_score'] - 
                               cease_trimmed['away_score'])

print(f"Total pitches: {len(cease_trimmed)}")
print("\n=== Cease Pitch Distribution ===")
print(cease_trimmed['pitch_type'].value_counts())